In [ ]:
from __future__ import annotations

import time
from typing import Any, Dict, Optional, Tuple

from kaggle.api.kaggle_api_extended import KaggleApi


def _as_dict(x: Any) -> Dict[str, Any]:
    """The Kaggle API sometimes returns a dict and sometimes a Submission object; normalize to dict."""
    if isinstance(x, dict):
        return x
    # Submission objects typically have __dict__ or attributes
    d = {}
    for k in dir(x):
        if k.startswith("_"):
            continue
        try:
            v = getattr(x, k)
        except Exception:
            continue
        # Filter out methods
        if callable(v):
            continue
        d[k] = v
    # If the object itself has __dict__, merge it in
    if hasattr(x, "__dict__"):
        d.update(getattr(x, "__dict__", {}))
    return d


def submit_and_get_public_score(
    competition: str,
    submission_path: str,
    message: str = "auto-submit",
    poll_interval_sec: int = 15,
    timeout_sec: int = 15 * 60,
) -> Tuple[Optional[str], Optional[float], Dict[str, Any]]:
    """
    Submit and retrieve the public score (if timeout, score is None).
    Returns: (submission_date, public_score, latest_submission_raw_dict)
    """
    api = KaggleApi()
    api.authenticate()

    # 1) Submit
    api.competition_submit(
        file_name=submission_path,
        message=message,
        competition=competition,
    )

    # 2) Poll submissions
    start = time.time()
    last_print = None

    while True:
        # Key fix: competition_submissions (not competitions_submissions)
        subs = api.competition_submissions(competition)
        if not subs:
            time.sleep(poll_interval_sec)
            continue

        latest_raw = subs[0]  # usually the newest
        latest = _as_dict(latest_raw)

        # Common field names: status, publicScore, date (vary by version/casing)
        status = str(latest.get("status", latest.get("Status", ""))).lower()
        public_score = latest.get("publicScore", latest.get("PublicScore", None))
        submission_date = latest.get("date", latest.get("Date", None))

        marker = (submission_date, status, public_score)
        if marker != last_print:
            print(f"[Kaggle] latest: date={submission_date} status={status} publicScore={public_score}")
            last_print = marker

        # Sometimes publicScore is a string
        if public_score not in (None, "", "null"):
            try:
                return str(submission_date) if submission_date is not None else None, float(public_score), latest
            except Exception:
                return str(submission_date) if submission_date is not None else None, None, latest

        if time.time() - start > timeout_sec:
            return str(submission_date) if submission_date is not None else None, None, latest

        time.sleep(poll_interval_sec)


# if __name__ == "__main__":


# dt, score, raw = submit_and_get_public_score(COMP, FILE, MSG, poll_interval_sec=20, timeout_sec=20*60)
# print("\n===== RESULT =====")
# print("submission_date:", dt)
# print("public_score:", score)
# print("raw_keys:", sorted(raw.keys()))


In [30]:
api = KaggleApi()
api.authenticate()
# message: str = "auto-submit",
poll_interval_sec: int = 15
timeout_sec: int = 15 * 60
# 1) Submit
competition = "churn-prediction-25-26"
submission_path = "submission.csv"
message = "local python auto submit"
api.competition_submit(
    file_name=submission_path,
    message=message,
    competition=competition,
)


100%|██████████| 28.4k/28.4k [00:00<00:00, 48.5kB/s]


{"message": "Successfully submitted to Churn prediction 25/26", "ref": 49002949}

In [31]:

# 2) Poll submissions
start = time.time()
last_print = None
subs = api.competition_submissions(competition)
# time.sleep(poll_interval_sec)
latest_raw = subs[0]  


In [26]:
import pandas as pd


In [1]:
# demo_submit.py
from kaggle_submit import submit_and_track

if __name__ == "__main__":
    info = submit_and_track(
        submission_path="submission.csv",            # File to upload
        competition="churn-prediction-25-26",       # Kaggle competition slug
        message="local python auto submit_ v1",         # Submission description
        wait_for_result=True,                       # Wait for the public score
        poll_interval=30,                           # Poll interval (seconds)
        max_polls=8,                                # Initial max polls
        log_path="submission_log.csv",              # Log file path
        extra_meta={"best_threshold": 0.53},        # Optional: extra metadata
    )
    print("Final submission info:", info)


Submitting submission.csv to churn-prediction-25-26 with note: local python auto submit_ v1


100%|██████████| 28.4k/28.4k [00:00<00:00, 60.9kB/s]


Poll 1/8: status=complete score=0.61147
Final submission info: {'ref': 49003577, 'description': 'local python auto submit_ v1', 'status': 'complete', 'score': 0.61147, 'fileName': 'submission.csv', 'submissionDate': datetime.datetime(2025, 12, 12, 17, 21, 11, 157000), 'error': '', 'teamName': 'Yuhan Wu', 'submittedBy': 'yuhanwu1003', 'isFrozen': True}


In [39]:
(
    latest_raw.public_score  is not None
    and latest_raw.public_score  != ''
)

False

In [ ]:
latest_raw.public_score is not None and is not ''


# while True:
#     # Key fix: competition_submissions (not competitions_submissions)
#     subs = api.competition_submissions(competition)
#     if not subs:
#         time.sleep(poll_interval_sec)
#         continue

#     latest_raw = subs[0]  # usually the newest
#     latest = _as_dict(latest_raw)

#     # Common field names: status, publicScore, date (vary by version/casing)
#     status = str(latest.get("status", latest.get("Status", ""))).lower()
#     public_score = latest.get("publicScore", latest.get("PublicScore", None))
#     submission_date = latest.get("date", latest.get("Date", None))

#     marker = (submission_date, status, public_score)
#     if marker != last_print:
#         print(f"[Kaggle] latest: date={submission_date} status={status} publicScore={public_score}")
#         last_print = marker

#     # Sometimes publicScore is a string
#     if public_score not in (None, "", "null"):
#         try:
#             return str(submission_date) if submission_date is not None else None, float(public_score), latest
#         except Exception:
#             return str(submission_date) if submission_date is not None else None, None, latest

#     if time.time() - start > timeout_sec:
#         return str(submission_date) if submission_date is not None else None, None, latest

#     time.sleep(poll_interval_sec)


SyntaxError: invalid syntax (1433061293.py, line 1)